# Part 7A: S&P 500 Price Data Download (Colab → Drive)

**목적:** Colab에서 Yahoo Finance 가격 데이터를 다운로드하여 Drive에 pickle로 저장

1. S&P 500 구성종목 수집 (Wikipedia)
2. 전 종목 가격 데이터 다운로드 + 보조지표 계산
3. pickle로 Drive 저장 (종목별 체크포인트)

**이후:** 로컬에서 `run_part7_local.py`로 이미지 병렬 생성

In [ ]:
# Cell 1: Install & Imports
!pip install -q yfinance

import os, json, time, warnings, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from collections import Counter
from io import StringIO
import requests

print('All imports OK')

In [ ]:
# Cell 2: Mount Drive & Paths

from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    'start_date': '2010-01-01',
}

DRIVE_BASE = Path('/content/drive/MyDrive/KAIST_experiment/cross_sectional/')
META_DIR = DRIVE_BASE / 'metadata'
DATA_DIR = DRIVE_BASE / 'price_data'
for d in [META_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Drive base: {DRIVE_BASE}')
print(f'Price data: {DATA_DIR}')
print('Ready')

In [ ]:
# Cell 3: Fetch S&P 500 Constituents

def get_sp500_constituents():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; KAIST-Research/1.0)'}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    tables = pd.read_html(StringIO(resp.text))
    df = tables[0]
    df['Symbol'] = df['Symbol'].str.replace('.', '-', regex=False)
    constituents = []
    for _, row in df.iterrows():
        constituents.append({
            'ticker': row['Symbol'],
            'company': row['Security'],
            'sector': row['GICS Sector'],
            'sub_industry': row['GICS Sub-Industry'],
            'date_added': str(row.get('Date added', '')),
        })
    return constituents

constituents_path = META_DIR / 'sp500_constituents.json'
if constituents_path.exists():
    with open(constituents_path) as f:
        constituents = json.load(f)
    print(f'Loaded {len(constituents)} constituents from cache')
else:
    constituents = get_sp500_constituents()
    with open(constituents_path, 'w') as f:
        json.dump(constituents, f, indent=2)
    print(f'Fetched {len(constituents)} constituents')

sector_counts = Counter(c['sector'] for c in constituents)
print(f'\n=== GICS Sectors ===')
for sector, count in sorted(sector_counts.items(), key=lambda x: -x[1]):
    print(f'  {sector:>30}: {count}')
print(f'  {"TOTAL":>30}: {len(constituents)}')

In [ ]:
# Cell 4: Download All Stocks → Pickle (with checkpoint + retry)

def download_stock_data(ticker, start_date):
    """단일 종목 가격 데이터 다운로드 + 보조지표 계산"""
    try:
        df = yf.download(ticker, start=start_date, progress=False)
        if df is None or len(df) < 200:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df['MA5'] = df['Close'].rolling(5).mean()
        df['MA60'] = df['Close'].rolling(60).mean()
        df['MA120'] = df['Close'].rolling(120).mean()
        ma20 = df['Close'].rolling(20).mean()
        std20 = df['Close'].rolling(20).std()
        df['BB_upper'] = ma20 + 2 * std20
        df['BB_lower'] = ma20 - 2 * std20
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['RSI'] = 100 - (100 / (1 + gain / loss))
        df = df.dropna()
        return df
    except Exception:
        return None

# Load checkpoint
status_path = META_DIR / 'download_status.json'
if status_path.exists():
    with open(status_path) as f:
        status = json.load(f)
else:
    status = {}

all_tickers = [c['ticker'] for c in constituents]
# Only retry tickers not yet OK
need = [t for t in all_tickers if status.get(t, {}).get('status') != 'ok']

print(f'Total: {len(all_tickers)}, Already OK: {len(all_tickers)-len(need)}, Remaining: {len(need)}')

t0 = time.time()
failed = []
for idx, ticker in enumerate(need):
    df = None
    for attempt in range(3):
        df = download_stock_data(ticker, CONFIG['start_date'])
        if df is not None:
            break
        time.sleep(3 * (attempt + 1))  # Backoff on rate limit

    if df is not None and len(df) >= 200:
        df.to_pickle(str(DATA_DIR / f'{ticker}.pkl'))
        status[ticker] = {'rows': len(df), 'status': 'ok'}
    else:
        status[ticker] = {'status': 'failed'}
        failed.append(ticker)

    # Rate limit protection
    time.sleep(0.5)

    # Checkpoint + progress every 10
    if (idx + 1) % 10 == 0 or idx == len(need) - 1:
        with open(status_path, 'w') as f:
            json.dump(status, f, indent=2)
        elapsed = time.time() - t0
        ok_so_far = sum(1 for v in status.values() if v.get('status') == 'ok')
        rate = (idx + 1) / elapsed * 60
        eta = (len(need) - idx - 1) / max(rate, 0.1)
        print(f'  [{idx+1}/{len(need)}] OK:{ok_so_far} Failed:{len(failed)} | {elapsed:.0f}s | {rate:.0f}/min | ETA {eta:.0f}min')

ok_tickers = [t for t, v in status.items() if v.get('status') == 'ok']
print(f'\n=== Download Complete ===')
print(f'OK: {len(ok_tickers)} / {len(all_tickers)}')
print(f'Failed: {len(failed)}')
if failed:
    print(f'Failed tickers: {failed[:20]}')

In [ ]:
# Cell 5: Verify & Summary

import os

pkl_files = list(DATA_DIR.glob('*.pkl'))
print(f'=== DRIVE VERIFICATION ===')
print(f'Pickle files saved: {len(pkl_files)}')

total_size = sum(f.stat().st_size for f in pkl_files)
print(f'Total size: {total_size / 1024 / 1024:.1f} MB')

print(f'\n=== 다음 단계 ===')
print(f'1. Drive에서 price_data/ 폴더를 로컬로 다운로드')
print(f'   경로: MyDrive/KAIST_experiment/cross_sectional/price_data/')
print(f'2. 로컬에서 python3 run_part7_local.py 실행 (이미지 병렬 생성)')
print(f'3. 생성된 이미지를 Drive에 업로드 후 Part 8 실행')